In [1]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# Assume X is your preprocessed data matrix (num_matches, num_features)
# For example, X.shape might be (28000, 200) if you have 28,000 matches with 200 features each.
input_dim = X.shape[1]
encoding_dim = 64  # Dimension of the latent space (you can adjust this)

# Define the autoencoder model architecture
input_layer = Input(shape=(input_dim,))
encoder = Dense(128, activation="relu")(input_layer)
encoder = Dense(encoding_dim, activation="relu")(encoder)

decoder = Dense(128, activation="relu")(encoder)
decoder = Dense(input_dim, activation="sigmoid")(decoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer=Adam(), loss="mse")

# Train the autoencoder (adjust epochs and batch_size as needed)
autoencoder.fit(X, X, epochs=50, batch_size=32, validation_split=0.1)

# Build a model to extract the latent features
encoder_model = Model(inputs=input_layer, outputs=encoder)
encoded_features = encoder_model.predict(X)


NameError: name 'X' is not defined

In [1]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Define a recursive function to flatten nested JSON ---
def flatten_json(nested_json, parent_key="", sep="."):
    """
    Recursively flattens a nested JSON/dictionary.
    For lists, each element’s index is appended to the key.
    """
    items = {}
    if isinstance(nested_json, dict):
        for key, value in nested_json.items():
            new_key = f"{parent_key}{sep}{key}" if parent_key else key
            items.update(flatten_json(value, new_key, sep=sep))
    elif isinstance(nested_json, list):
        for i, item in enumerate(nested_json):
            new_key = f"{parent_key}{sep}{i}"
            items.update(flatten_json(item, new_key, sep=sep))
    else:
        items[parent_key] = nested_json
    return items

# --- Load all flattened match files into a DataFrame ---
def load_all_flat_matches(matches_dir):
    flat_list = []
    for root, _, files in os.walk(matches_dir):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r") as f:
                        data = json.load(f)
                    flat = flatten_json(data)
                    flat_list.append(flat)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    return pd.DataFrame(flat_list)

# Change this path to where your JSON match files are stored.
matches_dir = "data-collection/matches"
df = load_all_flat_matches(matches_dir)
print("Initial DataFrame shape (matches x features):", df.shape)

# --- Select only numeric columns ---
numeric_df = df.select_dtypes(include=[np.number])
print("Numeric DataFrame shape:", numeric_df.shape)

# --- Impute missing values ---
# For example, fill missing values with the column mean.
numeric_df_imputed = numeric_df.fillna(numeric_df.mean())

# --- Cluster features (i.e. columns) based on their values across samples ---
# Transpose so that rows represent features and columns represent match observations.
features_df = numeric_df_imputed.T
print("Features DataFrame shape (features x matches):", features_df.shape)

# --- Standardize the features ---
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_df)

# --- Dimensionality Reduction via PCA ---
# Choose a number of principal components that capture most variance (adjust n_components as needed)
pca = PCA(n_components=50)
latent_features = pca.fit_transform(features_scaled)
print("Latent features shape:", latent_features.shape)

# --- Cluster the latent feature representations using KMeans ---
n_clusters = 20  # Adjust the number of clusters as desired
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(latent_features)

# --- Create a DataFrame mapping original feature names to clusters ---
df_clustered = pd.DataFrame({'feature': features_df.index, 'cluster': clusters})
print("\nClustered Features:")
print(df_clustered.groupby('cluster')['feature'].apply(list))


Initial DataFrame shape (matches x features): (4024, 3716)
Numeric DataFrame shape: (4024, 3469)
Features DataFrame shape (features x matches): (3469, 4024)
Latent features shape: (3469, 50)


C:\Users\timtu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)



Clustered Features:
cluster
0     [info.gameDuration, info.mapId, info.participa...
1          [info.gameCreation, info.gameStartTimestamp]
2                                         [info.gameId]
3     [info.participants.1.perks.styles.0.selections...
4     [info.participants.1.perks.styles.1.selections...
5     [info.participants.5.perks.styles.1.selections...
6     [info.participants.2.perks.styles.1.selections...
7     [info.participants.4.perks.styles.1.selections...
8     [info.participants.5.perks.styles.1.selections...
9     [info.participants.3.perks.styles.1.selections...
10    [info.participants.5.perks.styles.0.selections...
11    [info.participants.3.perks.styles.1.selections...
12    [info.participants.8.perks.styles.1.selections...
13    [info.participants.6.perks.styles.1.selections...
14    [info.participants.6.perks.styles.0.selections...
15                              [info.gameEndTimestamp]
16    [info.participants.0.physicalDamageDealt, info...
17    [info.partici

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Define a recursive function to flatten nested JSON ---
def flatten_json(nested_json, parent_key="", sep="."):
    """
    Recursively flattens a nested JSON/dictionary.
    For lists, each element’s index is appended to the key.
    """
    items = {}
    if isinstance(nested_json, dict):
        for key, value in nested_json.items():
            new_key = f"{parent_key}{sep}{key}" if parent_key else key
            items.update(flatten_json(value, new_key, sep=sep))
    elif isinstance(nested_json, list):
        for i, item in enumerate(nested_json):
            new_key = f"{parent_key}{sep}{i}"
            items.update(flatten_json(item, new_key, sep=sep))
    else:
        items[parent_key] = nested_json
    return items

# --- Load all flattened match files into a DataFrame ---
def load_all_flat_matches(matches_dir):
    flat_list = []
    for root, _, files in os.walk(matches_dir):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r") as f:
                        data = json.load(f)
                    flat = flatten_json(data)
                    flat_list.append(flat)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    return pd.DataFrame(flat_list)







In [ ]:
# Change this path to where your JSON match files are stored.
matches_dir = "data-collection/matches"
df = load_all_flat_matches(matches_dir)
print("Initial DataFrame shape (matches x features):", df.shape)

# --- Select only numeric columns ---
numeric_df = df.select_dtypes(include=[np.number])
print("Numeric DataFrame shape:", numeric_df.shape)

Initial DataFrame shape (matches x features): (4024, 3716)
Numeric DataFrame shape: (4024, 3469)


In [ ]:
# --- Impute missing values ---
# For example, fill missing values with the column mean.
numeric_df_imputed = numeric_df.fillna(numeric_df.mean())

# --- Cluster features (i.e. columns) based on their values across samples ---
# Transpose so that rows represent features and columns represent match observations.
features_df = numeric_df_imputed.T
print("Features DataFrame shape (features x matches):", features_df.shape)

Features DataFrame shape (features x matches): (3469, 4024)


In [ ]:
# --- Standardize the features ---
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_df)

# --- Dimensionality Reduction via PCA ---
# Choose a number of principal components that capture most variance (adjust n_components as needed)
pca = PCA(n_components=50)
latent_features = pca.fit_transform(features_scaled)
print("Latent features shape:", latent_features.shape)

Latent features shape: (3469, 50)


In [ ]:
# --- Cluster the latent feature representations using KMeans ---
n_clusters = 20  # Adjust the number of clusters as desired
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(latent_features)

# --- Create a DataFrame mapping original feature names to clusters ---
df_clustered = pd.DataFrame({'feature': features_df.index, 'cluster': clusters})
print("\nClustered Features:")
print(df_clustered.groupby('cluster')['feature'].apply(list))


Clustered Features:
cluster
0     [info.gameDuration, info.mapId, info.participa...
1          [info.gameCreation, info.gameStartTimestamp]
2                                         [info.gameId]
3     [info.participants.5.perks.styles.1.selections...
4     [info.participants.1.perks.styles.1.selections...
5     [info.participants.4.perks.styles.1.selections...
6     [info.participants.2.perks.styles.1.selections...
7     [info.participants.3.perks.styles.1.selections...
8     [info.participants.1.perks.styles.0.selections...
9     [info.participants.5.perks.styles.0.selections...
10    [info.participants.3.perks.styles.1.selections...
11    [info.participants.5.perks.styles.1.selections...
12    [info.participants.6.perks.styles.1.selections...
13    [info.participants.8.perks.styles.1.selections...
14    [info.participants.6.perks.styles.0.selections...
15                              [info.gameEndTimestamp]
16    [info.participants.0.physicalDamageDealt, info...
17    [info.partici

C:\Users\timtu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [ ]:
# Instead of just doing print(df_clustered.groupby('cluster')['feature'].apply(list)),
# you can do something like this:

grouped = df_clustered.groupby('cluster')['feature']
for cluster_id, features in grouped:
    print(f"\nCluster {cluster_id}:")
    for feature in features:
        print("  ", feature)



Cluster 0:
   info.gameDuration
   info.mapId
   info.participants.0.PlayerScore0
   info.participants.0.PlayerScore1
   info.participants.0.PlayerScore10
   info.participants.0.PlayerScore11
   info.participants.0.PlayerScore2
   info.participants.0.PlayerScore3
   info.participants.0.PlayerScore4
   info.participants.0.PlayerScore5
   info.participants.0.PlayerScore6
   info.participants.0.PlayerScore7
   info.participants.0.PlayerScore8
   info.participants.0.PlayerScore9
   info.participants.0.allInPings
   info.participants.0.assistMePings
   info.participants.0.assists
   info.participants.0.baronKills
   info.participants.0.basicPings
   info.participants.0.bountyLevel
   info.participants.0.challenges.12AssistStreakCount
   info.participants.0.challenges.HealFromMapSources
   info.participants.0.challenges.InfernalScalePickup
   info.participants.0.challenges.SWARM_DefeatAatrox
   info.participants.0.challenges.SWARM_DefeatBriar
   info.participants.0.challenges.SWARM_DefeatMi

In [3]:
df.fillna(0, inplace=True)
numeric_df = df.select_dtypes(include=[np.number])
X = numeric_df.values.astype("float32")
print("Shape of X:", X.shape)


Shape of X: (4024, 3469)


In [35]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# Assume X is your flattened data as a NumPy array with shape (num_matches, num_features)
input_dim = X.shape[1]
encoding_dim = 64  # You can adjust the size of the latent space

# Build the autoencoder architecture
input_layer = Input(shape=(input_dim,))
# Encoder
encoded = Dense(128, activation="relu")(input_layer)
encoded = Dense(encoding_dim, activation="relu")(encoded)
# Decoder
decoded = Dense(128, activation="relu")(encoded)
decoded = Dense(input_dim, activation="sigmoid")(decoded)

# Construct and compile the autoencoder model
autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer=Adam(), loss="mse")

# Train the autoencoder
autoencoder.fit(X, X, epochs=50, batch_size=32, validation_split=0.1)

# Build the encoder model to extract latent features
encoder_model = Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder_model.predict(X)
print("Encoded features shape:", X_encoded.shape)


Epoch 1/50
114/114 [==============================] - 1s 7ms/step - loss: 2614471595452650225664.0000 - val_loss: 2618881463912776073216.0000
Epoch 2/50
114/114 [==============================] - 1s 6ms/step - loss: 2614472721352557068288.0000 - val_loss: 2618881463912776073216.0000
Epoch 3/50
114/114 [==============================] - 1s 6ms/step - loss: 2614472158402603646976.0000 - val_loss: 2618881463912776073216.0000
Epoch 4/50
114/114 [==============================] - 1s 6ms/step - loss: 2614472158402603646976.0000 - val_loss: 2618881463912776073216.0000
Epoch 5/50
114/114 [==============================] - 1s 6ms/step - loss: 2614472158402603646976.0000 - val_loss: 2618881463912776073216.0000
Epoch 6/50
114/114 [==============================] - 1s 6ms/step - loss: 2614472439877580357632.0000 - val_loss: 2618881463912776073216.0000
Epoch 7/50
114/114 [==============================] - 1s 6ms/step - loss: 2614472158402603646976.0000 - val_loss: 2618881463912776073216.0000
Epoch 

KeyboardInterrupt: 

In [37]:
# Assuming df is your DataFrame containing the flattened JSON data
df.to_csv("flattened_matches.csv", index=False)
